In [1]:
import pandas as pd, numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestClassifier
from joblib import dump

# -------------- data ---------------------------------------------------
expr = pd.read_csv("data/GENE_MATRIX_COMBO_MAR31.csv", index_col=0).T
meta = pd.read_csv("data/METADATA_COMBO_MAR31_covarsRemoved.csv", index_col=0)
expr = expr.loc[meta.index]
y      = (meta["condition"] == "RIF").astype(int).values
groups = meta["study"].values

print(f"samples={expr.shape[0]} genes={expr.shape[1]} classes={np.bincount(y)}")

# -------------- selector -----------------------------------------------
class ComboSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k=30): self.k = k
    def fit(self, X, y):
        X_ = X.values if isinstance(X, pd.DataFrame) else X
        f  = f_classif(X_, y)[0]
        mi = mutual_info_classif(X_, y, random_state=0)
        keep = np.argsort(f + mi)[-self.k:]
        self.mask_ = np.zeros(X_.shape[1], dtype=bool)
        self.mask_[keep] = True
        return self
    def transform(self, X):
        X_ = X.values if isinstance(X, pd.DataFrame) else X
        return X_[:, self.mask_]
    def get_support(self): return self.mask_

numeric_pipe = Pipeline([
    ("select", ComboSelector(k=30)),
    ("scale",  StandardScaler())
])

preprocess = ColumnTransformer([("genes", numeric_pipe, expr.columns)])

# -------------- model & CV ---------------------------------------------
rf   = RandomForestClassifier(n_estimators=500,
                              class_weight="balanced",
                              random_state=42)
pipe = Pipeline([("prep", preprocess), ("clf", rf)])

param_grid = {"clf__max_depth": [None, 10, 20]}
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer = GroupKFold(n_splits=5)
gs    = GridSearchCV(pipe, param_grid, cv=inner,
                     scoring="roc_auc", n_jobs=-1)

cv = cross_validate(gs, expr, y,
                    cv=outer, groups=groups,
                    scoring="roc_auc",
                    return_estimator=True, n_jobs=-1)

print(f"\nLeak-free AUROC = {cv['test_score'].mean():.3f} ± {cv['test_score'].std():.3f}")

kept = [est.best_estimator_.named_steps["prep"]
                      .named_transformers_["genes"]
                      .named_steps["select"].get_support().sum()
        for est in cv["estimator"]]
print("genes kept per fold:", kept)

best = max(cv["estimator"], key=lambda g: g.best_score_)
dump(best.best_estimator_, "rf_30gene_leakfree.joblib")
print("\nModel saved to rf_30gene_leakfree.joblib")

samples=217 genes=10489 classes=[121  96]

Leak-free AUROC = 0.607 ± 0.219
genes kept per fold: [np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30)]

Model saved to rf_30gene_leakfree.joblib
